# Lecture 4: SciPy fitting, uncertainty, residuals, covariance, and least squares

**PHYS690: Computational Methods for Physics Research**  
**Tuesday, September 8, 2026**

Today we connect measurement uncertainty to least-squares fitting. The theme is practical: once you have data points with uncertainties, how do you compare them with a model, estimate model parameters, and report the uncertainty and covariance of those parameters?

We will use:

- **NumPy** for arrays and vectorized model calculations.
- **pandas** for compact fit-result tables.
- **Matplotlib** for data, residual, chi-square, and covariance plots.
- **SciPy** for `scipy.optimize.curve_fit`.
- **Git/GitHub** for saving and pushing your local notebook work.

About one third of today will be hands-on coding or terminal workflow practice. When you see an **In-class coding activity**, pause and complete the notebook or terminal task yourself before we discuss.

## How to use this notebook

This notebook is intended to run on your laptop in **VS Code**, not Google Colab.

Before running the notebook, open this repository in VS Code and activate your course environment in the VS Code terminal.

On macOS/Linux:

```bash
source .venv/bin/activate
```

On Windows using Git Bash:

```bash
source .venv/Scripts/activate
```

If you have not installed the packages yet, run this while the environment is active:

```bash
pip install numpy scipy matplotlib pandas ipykernel jupyter
```

Then select the `.venv` Python kernel in VS Code. This notebook creates files under `scratch/lecture04/`, which is intentionally ignored by Git in this course repository.

## Resources for today

- SciPy `curve_fit`: [official documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)
- SciPy `least_squares`: [official documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html)
- NumPy: [absolute beginner's guide](https://numpy.org/doc/stable/user/absolute_beginners)
- Matplotlib: [getting started guide](https://matplotlib.org/stable/users/getting_started/)
- GitHub Docs: [pushing commits to a remote repository](https://docs.github.com/en/get-started/using-git/pushing-commits-to-a-remote-repository)

The goal is not to memorize the documentation. The goal is to recognize the ingredients of a fit and know where to check details when you need them.

## Learning goals

By the end of this lecture, you should be able to:

- Distinguish variance, standard deviation, standard error, and measurement uncertainty.
- Compute residuals and normalized residuals between data and a model.
- Explain why weighted least squares minimizes $\chi^2 = \sum_i [(y_i - f(x_i;\theta))/\sigma_i]^2$.
- Use `scipy.optimize.curve_fit` with `p0`, `sigma`, and `absolute_sigma=True`.
- Read parameter uncertainties and parameter covariance from the covariance matrix returned by `curve_fit`.
- Recognize when two fit parameters are correlated.
- Commit and push notebook changes to a username branch on GitHub.

# Part 1: Imports and scratch workspace

A fitting workflow needs numerical arrays, plots, and a tested optimization routine. We also create a scratch directory for generated data and figures.

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import subprocess
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import curve_fit

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "lectures":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd()

SCRATCH_DIR = PROJECT_ROOT / "scratch" / "lecture04"
DATA_DIR = SCRATCH_DIR / "data"
FIGURE_DIR = SCRATCH_DIR / "figures"

for directory in [DATA_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(seed=6904)

print(f"Python executable: {sys.executable}")
print(f"Working directory: {PROJECT_ROOT}")
print(f"NumPy {np.__version__} | pandas {pd.__version__}")
print(f"SciPy {scipy.__version__} | Matplotlib {matplotlib.__version__}")

### In-class coding activity 1: verify SciPy in your environment, 3 minutes

In the VS Code terminal, with your `.venv` active, run:

```bash
python -c "import scipy; print(scipy.__version__)"
pwd
ls scratch/lecture04
```

Compare the terminal Python environment with the notebook output above.

# Part 2: Variance, uncertainty, and what an error bar means

Repeated measurements of the same quantity usually do not give exactly the same value. Some spread comes from counting statistics, detector resolution, electronic noise, calibration uncertainty, or real physical variation.

A few useful quantities:

| Quantity | Meaning | Typical use |
| --- | --- | --- |
| Variance $s^2$ | Average squared spread around the mean | Describes scatter in repeated values |
| Standard deviation $s$ | Square root of variance | Typical spread of individual measurements |
| Measurement uncertainty $\sigma_i$ | Expected one-standard-deviation uncertainty on point $i$ | Used to weight residuals in a fit |

Examples across physics:

- **Nuclear/particle physics:** If a detector counts $N$ independent events in a bin, the counting uncertainty is often approximated by $\sqrt{N}$. For a ratio or cross section, those counting uncertainties propagate into the plotted error bars.
- **Plasma physics:** Probe or interferometry measurements may have time-varying signals plus instrumental noise. Repeated samples or calibration data can be used to estimate an uncertainty on density, temperature, or fluctuation amplitude.

In least-squares fitting, the uncertainty tells the fit how seriously to take each residual. A point with small $\sigma_i$ carries more weight than a point with large $\sigma_i$.

In [ ]:
# Repeated measurements of one quantity.
measurements = np.array([1.02, 0.97, 1.01, 1.04, 0.99, 1.03, 0.98, 1.00])

mean_measurement = measurements.mean()
sample_variance = measurements.var(ddof=1)
sample_std = measurements.std(ddof=1)
standard_error = sample_std / np.sqrt(measurements.size)

pd.DataFrame(
    {
        "quantity": ["mean", "sample variance", "sample std", "standard error on mean"],
        "value": [mean_measurement, sample_variance, sample_std, standard_error],
    }
)

### In-class coding activity 2: uncertainty estimates, 5 minutes

Complete the cell below. Use code to estimate uncertainties for three common situations:

- A nuclear/particle counting bin with `N_events = 144`.
- A plasma diagnostic with repeated density measurements stored in `density_measurements`.

For each case, compute the central value and an approximate one-standard-deviation uncertainty.

In [ ]:
# Activity: complete the uncertainty estimates.

# Nuclear/particle counting example.
N_events = 144
# TODO: Set this to the central count value.
count_estimate = ...
# TODO: Estimate the one-standard-deviation counting uncertainty.
count_uncertainty = ...

# Plasma diagnostic repeated-measurement example, in units of 10^18 m^-3.
density_measurements = np.array([4.8, 5.1, 5.0, 4.9, 5.2, 5.1, 4.9, 5.0])
# TODO: Compute the mean density.
density_mean = ...
# TODO: Compute the sample standard deviation of the repeated measurements.
density_std = ...
# TODO: Compute the standard error on the mean density.
density_standard_error = ...

uncertainty_examples = pd.DataFrame(
    {
        "case": ["counting bin", "plasma density mean"],
        "estimate": [count_estimate, density_mean],
        "uncertainty": [count_uncertainty, density_standard_error],
    }
)

uncertainty_examples

# Part 3: Residuals, normalized residuals, and chi-square

Now return to the synthetic position measurement from Lectures 02 and 03:

$$x(t) = A e^{-t/\tau} + c + \epsilon.$$

We will use the same generating parameters as before: $A = 1.0$, $\tau = 3.0~\mathrm{s}$, $c = 0.0$, and a point-by-point measurement uncertainty of $\sigma_x = 0.025$.

For the first exercise, pretend that the amplitude and offset are known. Then the only unknown model parameter is the decay time $\tau$. This is still simple enough that we can scan possible $\tau$ values by hand and watch how $\chi^2$ changes.

A **residual** is the difference between a measured value and a model prediction:

$$r_i = y_i - f(x_i;\theta).$$

If point $i$ has measurement uncertainty $\sigma_i$, the **normalized residual** is

$$z_i = \frac{y_i - f(x_i;\theta)}{\sigma_i}.$$

A normalized residual answers: how many error bars away is the data point from the model?

The weighted least-squares objective is

$$\chi^2(\theta) = \sum_i \left(\frac{y_i - f(x_i;\theta)}{\sigma_i}\right)^2.$$

A good fit has residuals that look like noise: no obvious trend with the independent variable, and typical normalized residuals of order 1. A large systematic pattern in the residuals usually means the model is missing something, even if the plotted curve looks plausible.

In [ ]:
# Build the same synthetic position data set used in Lectures 02 and 03.
time_s = np.linspace(0.0, 10.0, 21)

amplitude_true = 1.0
tau_true = 3.00
offset_true = 0.00

clean_position = amplitude_true * np.exp(-time_s / tau_true) + offset_true
position_error = np.full_like(time_s, 0.025)
measured_position = clean_position + rng.normal(loc=0.0, scale=position_error)

position_data = pd.DataFrame(
    {
        "time_s": time_s,
        "position": measured_position,
        "position_error": position_error,
        "generating_model": clean_position,
    }
)

position_csv_path = DATA_DIR / "synthetic_position_measurement.csv"
position_data.to_csv(position_csv_path, index=False)
position_data.head()

In [ ]:
# Plot the synthetic position data set before computing chi-square.
fig, ax = plt.subplots(figsize=(7, 4))

ax.errorbar(
    position_data["time_s"],
    position_data["position"],
    yerr=position_data["position_error"],
    fmt="o",
    capsize=3,
    label="synthetic measurements",
)
ax.plot(position_data["time_s"], position_data["generating_model"], label="generating model")
ax.set_xlabel("time (s)")
ax.set_ylabel("position")
ax.set_title("Synthetic exponential relaxation data set")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

position_data_plot_path = FIGURE_DIR / "synthetic_position_dataset.png"
fig.savefig(position_data_plot_path)

print(f"Saved figure to: {position_data_plot_path}")

In [ ]:
def fixed_amplitude_decay_model(t, tau):
    """Return A exp(-t/tau) + c with amplitude and offset fixed."""
    return amplitude_true * np.exp(-t / tau) + offset_true


def chi_square_for_tau(tau, data):
    """Return chi-square for the one-parameter decay-time model."""
    model_position = fixed_amplitude_decay_model(data["time_s"], tau)
    residuals = data["position"] - model_position
    normalized_residuals = residuals / data["position_error"]
    return np.sum(normalized_residuals**2)

starting_tau = 2.0
starting_model = fixed_amplitude_decay_model(position_data["time_s"], starting_tau)
starting_residuals = position_data["position"] - starting_model
starting_normalized_residuals = starting_residuals / position_data["position_error"]
starting_chi2 = chi_square_for_tau(starting_tau, position_data)

print(f"Starting tau: {starting_tau:.3f} s")
print(f"Starting chi-square: {starting_chi2:.2f}")


### In-class coding activity 3: watch $\chi^2$ change, 5 minutes

Complete the cell below to scan many possible decay-time values. Your plot should show $\chi^2$ as a function of $\tau$ and mark the $\tau$ value with the smallest $\chi^2$.

This is the simplest possible version of least-squares fitting: try parameter values, compute the residuals for each one, square the normalized residuals, and find the parameter value where the sum is smallest.

In [ ]:
# Activity: scan tau values and compute chi-square for each one.
tau_grid = np.linspace(1.0, 5.0, 200)
chi2_grid = []

for tau in tau_grid:
    # TODO: Compute chi-square for this trial tau using chi_square_for_tau(tau, position_data).
    chi2_value = ...
    chi2_grid.append(chi2_value)

chi2_grid = np.array(chi2_grid)

# TODO: Find the index where chi-square is smallest. (Hint: use np.argmin)
best_index = ...
# TODO: Use best_index to get the best tau from tau_grid.
best_tau_scan = ...
# TODO: Use best_index to get the minimum chi-square from chi2_grid.
best_chi2_scan = ...

fig, ax = plt.subplots(figsize=(7, 4))
# TODO: Plot chi-square versus tau.
ax.plot(..., ...)
ax.set_xlabel("tau (s)")
ax.set_ylabel(r"$\chi^2$")
ax.set_title(r"One-parameter scan: $\chi^2$ versus decay time")
ax.legend()
fig.tight_layout()

chi2_scan_path = FIGURE_DIR / "one_parameter_tau_chi2_scan.png"
fig.savefig(chi2_scan_path)

print(f"Best scan tau: {best_tau_scan:.3f} s")
print(f"Best scan chi-square: {best_chi2_scan:.2f}")


In [ ]:
# Plot the data, scan-best model, and normalized residuals.
scan_model = fixed_amplitude_decay_model(position_data["time_s"], best_tau_scan)
scan_normalized_residuals = (position_data["position"] - scan_model) / position_data["position_error"]

fig, axes = plt.subplots(2, 1, figsize=(7, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

axes[0].errorbar(position_data["time_s"], position_data["position"], yerr=position_data["position_error"], fmt="o", capsize=3, label="data")
axes[0].plot(position_data["time_s"], scan_model, label=f"scan model: tau={best_tau_scan:.3f} s")
axes[0].set_ylabel("position")
axes[0].legend()

axes[1].axhline(0.0, color="0.5", lw=1)
axes[1].errorbar(position_data["time_s"], scan_normalized_residuals, yerr=np.ones(len(position_data)), fmt="o", capsize=3)
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("normalized residual")

fig.suptitle("Data and residuals for the one-parameter decay model")
fig.tight_layout()

residual_path = FIGURE_DIR / "one_parameter_tau_residuals.png"
fig.savefig(residual_path)


# Part 4: Using `scipy.optimize.curve_fit`

`curve_fit` automates the least-squares search. According to the SciPy documentation, the model function should take the independent variable first, followed by the fit parameters. In symbols, SciPy assumes something like

$$y_i = f(x_i; \theta_1, \theta_2, \ldots) + \epsilon_i.$$

The typical call is:

```python
popt, pcov = curve_fit(
    model_function,
    xdata,
    ydata,
    p0=initial_guess,
    sigma=y_uncertainties,
    absolute_sigma=True,
)
```

Important pieces:

- `model_function`: a Python function such as `f(x, theta_1, theta_2, ...)` with parameters $\vec{\theta}$
- `xdata`: the measured independent-variable values.
- `ydata`: the measured dependent-variable values.
- `p0`: initial guesses for $\vec{\theta}$ parameters. Nonlinear fits may depend on starting values.
- `sigma`: one-standard-deviation uncertainties on the `ydata` points, $\sigma_i$.
- `absolute_sigma=True`: treat `sigma` as real absolute uncertainties, so `pcov` has the corresponding scale.
- `popt`: optimized parameter values.
- `pcov`: approximate parameter covariance matrix.

Under the hood, `curve_fit` repeatedly evaluates your model, computes residuals, and adjusts parameters to reduce the sum of squared residuals. With one-dimensional `sigma`, the minimized objective is

$$\chi^2 = \sum_i \left(\frac{y_i - f(x_i;\vec{\theta})}{\sigma_i}\right)^2.$$

Near a trial parameter vector, the algorithm asks: if I change each parameter slightly, how does the model prediction change? Those local derivatives form the **Jacobian** matrix.

For a data set with $N$ measured points and a model with $M$ fitted parameters, the model Jacobian is an $N \times M$ matrix:

$$
J_{ij} = \frac{\partial f(x_i; \vec{\theta})}{\partial \theta_j}.
$$

Each row corresponds to one data point. Each column corresponds to one fit parameter. For the two-parameter decay model used later, the columns are the derivatives with respect to $A$ and $\tau$. The same idea appears again in Part 6, where $J(t)$ is the one-row Jacobian, or gradient, used to propagate the parameter covariance to the model prediction at a single time value.

SciPy uses this Jacobian to decide how to update the parameters during the least-squares search. If you do not provide one, `curve_fit` estimates it numerically. See the SciPy documentation for [`curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html) and [`least_squares`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html) for the `jac` argument and its finite-difference options.

In weighted least squares, the curvature of $\chi^2$ near the minimum is approximately related to $J^T W J$, where $W$ contains the inverse variances $1/\sigma_i^2$. The covariance matrix is roughly the inverse of that curvature matrix. This is why shallow directions in $\chi^2$ correspond to large parameter uncertainties, and tilted valleys correspond to correlated parameters.

For unconstrained fits, SciPy commonly uses a Levenberg-Marquardt-style local least-squares algorithm. With bounds, it switches to algorithms designed for constrained least squares. The covariance matrix is estimated from the local curvature of the objective near the optimum, so it is most trustworthy when the model is reasonably linear near the best-fit point and the uncertainties are meaningful.


### 1D example: fit the one-parameter model with `curve_fit`

Now use `curve_fit` for the same one-parameter decay-time problem. In this worked example the solution is provided so you can compare the automated least-squares result with the $\chi^2$ scan you completed above.

In [ ]:
# Worked example: use curve_fit for the same one-parameter problem.
initial_guess = [2.0]

popt, pcov = curve_fit(
    fixed_amplitude_decay_model,
    position_data["time_s"],
    position_data["position"],
    p0=initial_guess,
    sigma=position_data["position_error"],
    absolute_sigma=True,
)

# Extract the fitted parameter and its uncertainty.
fit_tau = popt[0]
fit_tau_uncertainty = np.sqrt(pcov[0, 0])
fit_chi2 = chi_square_for_tau(fit_tau, position_data)
fit_ndf = len(position_data) - len(popt)

fit_result_one_parameter = pd.DataFrame(
    {
        "parameter": ["tau"],
        "estimate": [fit_tau],
        "curve_fit_uncertainty": [fit_tau_uncertainty],
        "true_value": [tau_true],
        "chi2": [fit_chi2],
        "ndf": [fit_ndf],
    }
)

print(f"Scan tau: {best_tau_scan:.4f} s")
print(f"curve_fit tau: {fit_tau:.4f} +/- {fit_tau_uncertainty:.4f} s")
fit_result_one_parameter


### Parameter uncertainty from $\Delta\chi^2$

For a one-parameter fit, the one-standard-deviation uncertainty can be read from the points where

$$\Delta\chi^2 = \chi^2(\tau) - \chi^2_\mathrm{min} = 1.$$

This is the scan-based version of the uncertainty estimate that `curve_fit` reports from the covariance matrix. If the $\chi^2$ curve is close to parabolic near its minimum, the two approaches should agree well.

In [ ]:
# Worked example: compare curve_fit uncertainty with the Delta chi-square = 1 rule.
# Use a finer grid than the activity scan so the crossings are easier to locate.
fine_tau_grid = np.linspace(fit_tau - 5 * fit_tau_uncertainty, fit_tau + 5 * fit_tau_uncertainty, 1000)
fine_chi2_grid = np.array([
    chi_square_for_tau(tau, position_data)
    for tau in fine_tau_grid
])

minimum_index = np.argmin(fine_chi2_grid)
chi2_minimum = fine_chi2_grid[minimum_index]
delta_chi2_grid = fine_chi2_grid - chi2_minimum

# Find Delta chi-square = 1 crossings on the left and right of the minimum.
left_tau_values = fine_tau_grid[: minimum_index + 1]
left_delta_values = delta_chi2_grid[: minimum_index + 1]
right_tau_values = fine_tau_grid[minimum_index:]
right_delta_values = delta_chi2_grid[minimum_index:]

# np.interp expects the x-values to be increasing. On the left side, Delta chi-square
# decreases toward the minimum, so reverse that side before interpolating.
tau_left_1sigma = np.interp(1.0, left_delta_values[::-1], left_tau_values[::-1])
tau_right_1sigma = np.interp(1.0, right_delta_values, right_tau_values)

scan_uncertainty_minus = fit_tau - tau_left_1sigma
scan_uncertainty_plus = tau_right_1sigma - fit_tau
scan_uncertainty_average = 0.5 * (scan_uncertainty_minus + scan_uncertainty_plus)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(fine_tau_grid, fine_chi2_grid, label=r"$\chi^2(\tau)$")
ax.axvline(fit_tau, color="black", linestyle="-", label=rf"best $\tau$ = {fit_tau:.3f} s")
ax.axvline(tau_left_1sigma, color="tab:red", linestyle="--", label=r"$\Delta\chi^2 = 1$")
ax.axvline(tau_right_1sigma, color="tab:red", linestyle="--")
ax.axhline(chi2_minimum + 1.0, color="tab:red", linestyle=":")
ax.set_xlabel(r"$\tau$ (s)")
ax.set_ylabel(r"$\chi^2$")
ax.set_ylim(bottom=0., top=chi2_minimum + 20.0)
ax.set_title(r"One-parameter uncertainty from $\Delta\chi^2 = 1$")
ax.legend()
fig.tight_layout()

delta_chi2_path = FIGURE_DIR / "one_parameter_tau_delta_chi2_uncertainty.png"
fig.savefig(delta_chi2_path)

uncertainty_comparison = pd.DataFrame(
    {
        "method": ["curve_fit covariance", "Delta chi-square scan"],
        "tau_estimate": [fit_tau, fit_tau],
        "minus_uncertainty": [fit_tau_uncertainty, scan_uncertainty_minus],
        "plus_uncertainty": [fit_tau_uncertainty, scan_uncertainty_plus],
        "average_uncertainty": [fit_tau_uncertainty, scan_uncertainty_average],
    }
)

print(f"Delta chi-square left crossing:  tau = {tau_left_1sigma:.4f} s")
print(f"Delta chi-square right crossing: tau = {tau_right_1sigma:.4f} s")
print(f"curve_fit uncertainty:           {fit_tau_uncertainty:.4f} s")
print(f"average scan uncertainty:        {scan_uncertainty_average:.4f} s")
uncertainty_comparison


# Part 5: Two-parameter fits and covariance

Now let two parameters vary in the same exponential relaxation model:

$$x(t) = A e^{-t/\tau} + c.$$

To keep the example focused, we will still hold the offset fixed at the same value used in Lectures 02 and 03, $c = 0.0$, and fit only the amplitude $A$ and decay time $\tau$.

The amplitude and decay time are often correlated. If $\tau$ changes, the curve decays more slowly or quickly; a compensating change in $A$ can keep part of the curve close to the data. The covariance matrix returned by `curve_fit` describes this relationship.

For two parameters, the covariance matrix is

$$
\mathrm{pcov} =
\begin{bmatrix}
\sigma_A^2 & \mathrm{cov}(A,\tau) \\
\mathrm{cov}(A,\tau) & \sigma_\tau^2
\end{bmatrix}.
$$

The diagonal entries are parameter variances. Their square roots are one-standard-deviation parameter uncertainties. The off-diagonal entry is the covariance. A dimensionless correlation coefficient is

$$\rho_{A\tau} = \frac{\mathrm{cov}(A,\tau)}{\sigma_A \sigma_\tau}.$$

If $\rho$ is near +1 or -1, the parameters are strongly correlated. If it is near 0, the linearized fit sees little correlation between them.

In [ ]:
def amplitude_tau_decay_model(t, amplitude, tau):
    """Return A exp(-t/tau) + c with offset fixed."""
    return amplitude * np.exp(-t / tau) + offset_true

popt_two, pcov_two = curve_fit(
    amplitude_tau_decay_model,
    position_data["time_s"],
    position_data["position"],
    p0=[0.9, 2.5],
    sigma=position_data["position_error"],
    absolute_sigma=True,
)

amplitude_fit, tau_fit = popt_two
amplitude_uncertainty, tau_uncertainty = np.sqrt(np.diag(pcov_two))
correlation = pcov_two[0, 1] / (amplitude_uncertainty * tau_uncertainty)

fit_summary = pd.DataFrame(
    {
        "parameter": ["amplitude", "tau"],
        "estimate": [amplitude_fit, tau_fit],
        "uncertainty": [amplitude_uncertainty, tau_uncertainty],
        "true_value": [amplitude_true, tau_true],
    }
)

print("Covariance matrix:")
print(pcov_two)
print(f"correlation coefficient rho = {correlation:.3f}")
fit_summary


### In-class coding activity 4: visualize two-parameter covariance, 10 minutes

Complete the grid scan below. Compute $\chi^2$ for many amplitude/$\tau$ pairs and draw contours of

$$\Delta\chi^2 = \chi^2(A,\tau) - \chi^2_\mathrm{min}.$$

The tilted contour shape shows that amplitude and decay time are correlated. Then compare the sign of the tilt with the sign of the covariance matrix off-diagonal entry.

**Reference: choosing the contour level**

The change in $\chi^2$ required for a 1-sigma, or 68.3%, confidence region depends on how many fit parameters are being shown together. Mathematically, the threshold is the quantile of a chi-square distribution with degrees of freedom equal to the number of parameters in the contour:

```python
from scipy.stats import chi2
delta_chi2 = chi2.ppf(0.6827, number_of_parameters)
```

| Number of fit parameters shown together | $\Delta\chi^2$ for 68.3% confidence | How to use it |
| --- | ---: | --- |
| 1 | 1.00 | One-parameter uncertainty interval |
| 2 | 2.30 | Two-parameter error ellipse or contour |
| 3 | 3.53 | Three-parameter joint confidence volume |
| 4 | 4.72 | Four-parameter joint confidence region |
| 5 | 5.89 | Five-parameter joint confidence region |

For this activity you are plotting amplitude and $\tau$ together, so the 68.3% joint confidence contour is $\Delta\chi^2 = 2.30$. The other contour levels in the code, 6.18 and 11.83, are the usual two-parameter 95.4% and 99.7% levels.

Useful references: SciPy's [`scipy.stats.chi2`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2.html) documentation describes how to compute these quantiles with `ppf`, and the [Particle Data Group Review of Particle Physics](https://pdg.lbl.gov/) includes a [statistics review](https://pdg.lbl.gov/2026/reviews/rpp2026-rev-statistics.pdf) discussing confidence regions from likelihood and $\chi^2$ contours.


In [ ]:
# Activity: scan a grid in amplitude and tau.
amplitude_values = np.linspace(amplitude_fit - 4 * amplitude_uncertainty, amplitude_fit + 4 * amplitude_uncertainty, 80)
tau_values = np.linspace(tau_fit - 4 * tau_uncertainty, tau_fit + 4 * tau_uncertainty, 80)

amplitude_grid_2d, tau_grid_2d = np.meshgrid(amplitude_values, tau_values)
chi2_2d = np.empty_like(amplitude_grid_2d)

for row in range(chi2_2d.shape[0]):
    for col in range(chi2_2d.shape[1]):
        # TODO: Read the trial amplitude from amplitude_grid_2d.
        trial_amplitude = ...
        # TODO: Read the trial tau from tau_grid_2d.
        trial_tau = ...
        # TODO: Evaluate the exponential model for this trial parameter pair using position_data["time_s"].
        trial_model = ...
        # TODO: Compute residuals using position_data["position"].
        trial_residuals = ...
        # TODO: Compute chi-square using position_data["position_error"].
        chi2_2d[row, col] = ...

# Convert chi-square to delta-chi-square by subtracting the minimum.
best_chi2_two = np.min(chi2_2d)
delta_chi2 = chi2_2d - best_chi2_two

fig, ax = plt.subplots(figsize=(6, 5))

# TODO: Draw contours of delta_chi2 as a function of amplitude and tau (1, 2 and 3 sigma contours)
contours = ax.contour(..., ..., ..., levels=[2.30, 6.18, 11.83])
ax.clabel(contours, inline=True, fontsize=9)
ax.plot(amplitude_fit, tau_fit, "o", color="black", label="curve_fit result")
ax.set_xlabel("amplitude")
ax.set_ylabel("tau (s)")
ax.set_title(r"Two-parameter covariance: $\Delta\chi^2$ contours")
ax.legend()
fig.tight_layout()

covariance_plot_path = FIGURE_DIR / "amplitude_tau_covariance_contours.png"
fig.savefig(covariance_plot_path)

print("pcov off-diagonal cov(amplitude, tau):", pcov_two[0, 1])
print("correlation coefficient:", correlation)


### Covariance as an error ellipse

Let's compare the two-dimensional $\chi^2$ surface from the box above with a visualization of what the covariance matrix from `curve_fit` already says about the fitted parameters.

For the two-parameter vector $\vec{p} = (A, \tau)$, the covariance matrix defines an ellipse through

$$
(\vec{p} - \vec{p}_\mathrm{best})^T C^{-1} (\vec{p} - \vec{p}_\mathrm{best}) = \Delta\chi^2.
$$

The eigenvectors of $C$ give the directions of the ellipse axes. The eigenvalues give the variances along those directions. For a joint 68% confidence region with two fitted parameters, the conventional contour uses $\Delta\chi^2 = 2.30$. The tilt of the ellipse is the visual signature of the amplitude-$\tau$ correlation.


*Step 1:* First collect the ingredients that define the ellipse: the best-fit point, the covariance matrix, and the confidence level. For a joint 68.3% region in two parameters, use $\Delta\chi^2 = 2.30$.


In [ ]:
# Step 1: collect the ingredients for the covariance ellipse.
from matplotlib.patches import Ellipse

parameter_center = np.array([amplitude_fit, tau_fit])
covariance_matrix = pcov_two
joint_delta_chi2_1sigma = 2.30

print("ellipse center [amplitude, tau]:", parameter_center)
print("covariance matrix:")
print(covariance_matrix)


*Step 2:* diagonalize the covariance matrix. The eigenvectors point along the principal axes of the ellipse. The eigenvalues are the variances along those rotated directions, so the full axis lengths are proportional to $2\sqrt{\Delta\chi^2\,\lambda}$.


In [ ]:
# Step 2: diagonalize the covariance matrix to get ellipse axes and orientation.
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

major_axis_vector = eigenvectors[:, 0]
ellipse_angle_degrees = np.degrees(np.arctan2(major_axis_vector[1], major_axis_vector[0]))
ellipse_width, ellipse_height = 2.0 * np.sqrt(joint_delta_chi2_1sigma * eigenvalues)

ellipse_geometry = pd.DataFrame(
    {
        "quantity": ["width", "height", "angle_degrees"],
        "value": [ellipse_width, ellipse_height, ellipse_angle_degrees],
    }
)
ellipse_geometry


*Step 3:* Finally draw the ellipse centered on the fitted amplitude and decay time. The black cross shows the one-dimensional parameter uncertainties from the diagonal elements of `pcov_two`; the blue ellipse shows the joint uncertainty region, including the covariance between parameters.


In [ ]:
# Step 3: draw the covariance error ellipse from the curve_fit covariance matrix.
fig, ax = plt.subplots(figsize=(6.5, 5))
ellipse = Ellipse(
    xy=parameter_center,
    width=ellipse_width,
    height=ellipse_height,
    angle=ellipse_angle_degrees,
    facecolor="tab:blue",
    edgecolor="tab:blue",
    alpha=0.18,
    lw=2,
    label=r"joint 68% ellipse, $\Delta\chi^2=2.30$",
)
ax.add_patch(ellipse)

ax.plot(amplitude_fit, tau_fit, "o", color="black", label="curve_fit result")
ax.errorbar(
    amplitude_fit,
    tau_fit,
    xerr=amplitude_uncertainty,
    yerr=tau_uncertainty,
    fmt="none",
    ecolor="black",
    capsize=4,
    label="1D parameter uncertainties",
)

ax.set_xlabel("amplitude")
ax.set_ylabel(r"$\tau$ (s)")
ax.set_title("Covariance ellipse from curve_fit")
ax.grid(alpha=0.25)
ax.legend(loc="upper left")

ax.autoscale_view()
ax.margins(0.20)
fig.tight_layout()

ellipse_plot_path = FIGURE_DIR / "amplitude_tau_covariance_ellipse.png"
fig.savefig(ellipse_plot_path)

print(f"Saved figure to: {ellipse_plot_path}")
print(f"ellipse angle: {ellipse_angle_degrees:.1f} degrees")
print(f"correlation coefficient: {correlation:.3f}")


# Part 6: Propagating fit uncertainty to the model curve

The covariance matrix does not only give uncertainties on the fitted parameters. It can also be propagated through the model to estimate the uncertainty on the model prediction itself.

For the two-parameter model

$$x(t; A, \tau) = A e^{-t/\tau} + c,$$

the central curve is evaluated at the best-fit parameters, $A_\mathrm{fit}$ and $\tau_\mathrm{fit}$. To draw an uncertainty band, ask how the model prediction changes when the fit parameters move within their covariance ellipse.

This uses the same Jacobian idea introduced in Part 4. In Part 4, the full fitting Jacobian has one row for every measured time value and one column for every fitted parameter:

$$
J_{ij} = \frac{\partial x(t_i; A, \tau)}{\partial p_j},
\quad \vec{p} = (A, \tau).
$$

For plotting the uncertainty band, we evaluate one row of this same Jacobian at each smooth time value $t$ along the model curve:

$$
J(t) = \begin{bmatrix}
\frac{\partial x(t; A, \tau)}{\partial A} &
\frac{\partial x(t; A, \tau)}{\partial \tau}
\end{bmatrix}_{A=A_\mathrm{fit},\,\tau=\tau_\mathrm{fit}}.
$$

This row vector tells us how sensitive the model prediction at that particular time is to small changes in $A$ and $\tau$. The propagated model variance at that time is approximately

$$
\sigma_x^2(t) = J(t)\, C\, J^T(t),
$$

where $C$ is the 2D parameter covariance matrix from `curve_fit`. In dimensions, this is

$$
(1 \times 2)(2 \times 2)(2 \times 1) = (1 \times 1),
$$

so the result is one variance for the model prediction for $x$ at given time $t$. Repeating this calculation for many time values gives the uncertainty band around the central model curve.

This calculation includes the amplitude uncertainty, the $\tau$ uncertainty, and their covariance. That last term matters: when two parameters are correlated, the uncertainty band can be larger or smaller than you would estimate by treating the parameters independently.

The plot below shows the measured data, the best-fit model curve, and the propagated 1-sigma uncertainty band on the model prediction. The lower subplot shows normalized residuals, with the same propagated model uncertainty converted into normalized-residual units. The band is not an uncertainty on each individual data point; it is the uncertainty in the fitted model curve caused by uncertainty in the fitted parameters.


*Step 1:* First build a smooth table of time values for the model curve. This is separate from the measured data table because the model should be drawn as a smooth line, not only at the measured points.


In [ ]:
# Step 1: build a smooth DataFrame for evaluating the fitted model curve.
smooth_model_data = pd.DataFrame(
    {
        "time_s": np.linspace(position_data["time_s"].min(), position_data["time_s"].max(), 300)
    }
)

# Evaluate the central model curve at the best-fit parameter values.
smooth_model_data["central_position"] = amplitude_tau_decay_model(
    smooth_model_data["time_s"],
    amplitude_fit,
    tau_fit,
)

smooth_model_data.head()


*Step 2:* Next compute the one-row Jacobian $J(t)$ at every smooth time value. These two columns say how much the model prediction would change for a small change in amplitude or a small change in $\tau$.


In [ ]:
# Step 2: compute the model derivatives with respect to the fitted parameters.
smooth_model_data["d_model_d_amplitude"] = np.exp(-smooth_model_data["time_s"] / tau_fit)
smooth_model_data["d_model_d_tau"] = (
    amplitude_fit
    * np.exp(-smooth_model_data["time_s"] / tau_fit)
    * smooth_model_data["time_s"]
    / tau_fit**2
)

smooth_model_data[["time_s", "d_model_d_amplitude", "d_model_d_tau"]].head()


*Step 3:* Now propagate the parameter covariance matrix through the model. The expression `np.einsum("ij,jk,ik->i", ...)` evaluates $J(t) C J^T(t)$ for every row of `smooth_model_data`. The square root gives the 1-sigma uncertainty on the fitted model curve.


In [ ]:
# Step 3: propagate the parameter covariance to the model prediction.
jacobian = smooth_model_data[["d_model_d_amplitude", "d_model_d_tau"]].to_numpy()
model_variance = np.einsum("ij,jk,ik->i", jacobian, pcov_two, jacobian)

smooth_model_data["model_uncertainty"] = np.sqrt(model_variance)
smooth_model_data["lower_1sigma"] = (
    smooth_model_data["central_position"] - smooth_model_data["model_uncertainty"]
)
smooth_model_data["upper_1sigma"] = (
    smooth_model_data["central_position"] + smooth_model_data["model_uncertainty"]
)

# Convert the model uncertainty into normalized-residual units for the lower panel.
smooth_model_data["position_error_for_normalization"] = np.interp(
    smooth_model_data["time_s"],
    position_data["time_s"],
    position_data["position_error"],
)
smooth_model_data["normalized_model_uncertainty"] = (
    smooth_model_data["model_uncertainty"] / smooth_model_data["position_error_for_normalization"]
)

smooth_model_data[["time_s", "central_position", "model_uncertainty", "normalized_model_uncertainty"]].head()


*Step 4* Before plotting the residuals, add the fit prediction and normalized residuals to the measured-data table. This keeps the residual calculation tied to the same `position_data` DataFrame used for the measured points and error bars.


In [ ]:
# Step 4: compute residuals at the measured data points.
position_data["fit_position"] = amplitude_tau_decay_model(
    position_data["time_s"],
    amplitude_fit,
    tau_fit,
)
position_data["residual"] = position_data["position"] - position_data["fit_position"]
position_data["normalized_residual"] = position_data["residual"] / position_data["position_error"]

position_data[["time_s", "position", "fit_position", "residual", "normalized_residual"]].head()


*Step 5:* Finally make the diagnostic figure. The upper panel shows the fitted curve and its propagated uncertainty band in position units. The lower panel shows normalized residuals, with the propagated model uncertainty band converted into the same normalized units.


In [ ]:
# Step 5: plot the fitted model, propagated uncertainty band, and normalized residuals.
fig, axes = plt.subplots(
    2,
    1,
    figsize=(7, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},
)

axes[0].errorbar(
    position_data["time_s"],
    position_data["position"],
    yerr=position_data["position_error"],
    fmt="o",
    capsize=3,
    label="data",
)
axes[0].plot(
    smooth_model_data["time_s"],
    smooth_model_data["central_position"],
    color="black",
    label="best-fit model",
)
axes[0].fill_between(
    smooth_model_data["time_s"],
    smooth_model_data["lower_1sigma"],
    smooth_model_data["upper_1sigma"],
    color="tab:blue",
    alpha=0.25,
    label="propagated model uncertainty",
)
axes[0].set_ylabel("position")
axes[0].set_title("Two-parameter fit with propagated model uncertainty")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].fill_between(
    smooth_model_data["time_s"],
    -smooth_model_data["normalized_model_uncertainty"],
    smooth_model_data["normalized_model_uncertainty"],
    color="tab:blue",
    alpha=0.18,
    label="propagated model uncertainty",
)
axes[1].axhline(0.0, color="0.5", lw=1)
axes[1].errorbar(
    position_data["time_s"],
    position_data["normalized_residual"],
    yerr=np.ones(len(position_data)),
    fmt="o",
    capsize=3,
)
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("normalized residual")
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.tight_layout()

fit_band_plot_path = FIGURE_DIR / "amplitude_tau_fit_uncertainty_band.png"
fig.savefig(fit_band_plot_path)

print(f"Saved figure to: {fit_band_plot_path}")


# Part 8: Putting the tools together

A fitting workflow often looks like this:

| Step | Tool or idea | Example from today |
| --- | --- | --- |
| Estimate uncertainty | statistics, counting rules, repeated measurements | $\sqrt{N}$, standard error, shot noise |
| Propose a model | physics reasoning | `fixed_amplitude_decay_model`, `amplitude_tau_decay_model` |
| Compare model to data | residuals | `y - model_y` |
| Weight by uncertainty | normalized residuals | `(y - model_y) / sigma_y` |
| Minimize mismatch | least squares | minimize $\chi^2$ |
| Estimate parameter uncertainty | covariance matrix | `np.sqrt(np.diag(pcov))` |
| Check model quality | residual plots | residuals versus time |
| Preserve workflow | Git/GitHub | branch, commit, push |

The numerical tools are important, but the scientific judgment is just as important: uncertainty estimates, residual patterns, and parameter covariance all affect what you can honestly claim from a fit.
